# Домашнее задание по теме «Базовые принципы ML и метрики регрессии»

Это первое домашнее задание на курсе по ML.

<!-- Повторим ключевые особенности, о которых стоит помнить (со многими из них ты уже сталкивался в рамках курса "Искуственный интеллект"): -->




Важно помнить:
* За каждую задачу ты получаешь баллы.
* Сумма баллов за все задания — 12.
Максимальный балл за домашнее задание — 10. Разница в 2 балла позволит выбирать задачи или решить все, но получить максимальный балл даже в случае ошибки.
* В заданиях будут подсказки, инструкции и дополнительная теория, которая поможет по-новому посмотреть на тему.
* Условия задач — это техническое задание для разработчика, которому надо точно следовать.



## Импорт библиотек

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

## Загрузка данных

In [ ]:
!gdown 13vRyN_RKtoUWtGWi0O8ZlP5Q2C0O0F9l

Downloading...
From: https://drive.google.com/uc?id=13vRyN_RKtoUWtGWi0O8ZlP5Q2C0O0F9l
To: /content/store_sharing.csv
100% 1.03M/1.03M [00:00<00:00, 122MB/s]


Дальше ты будешь работать с датасетом, который содержит данные по аренде велосипедов. Ниже подробно описаны его признаки.

In [ ]:
bike = pd.read_csv('/content/store_sharing.csv')
bike.head()

,timestamp,cnt,t1,t2,hum,wind_speed,weather_code,is_holiday,is_weekend,season
0,2015-01-04 00:00:00,182,3.0,2.0,93.0,6.0,3.0,0.0,1.0,3.0
1,2015-01-04 01:00:00,138,3.0,2.5,93.0,5.0,1.0,0.0,1.0,3.0
2,2015-01-04 02:00:00,134,2.5,2.5,96.5,0.0,1.0,0.0,1.0,3.0
3,2015-01-04 03:00:00,72,2.0,2.0,100.0,0.0,1.0,0.0,1.0,3.0
4,2015-01-04 04:00:00,47,2.0,0.0,93.0,6.5,1.0,0.0,1.0,3.0


**Признаки**

* `timestamp` — временная метка, используется для группировки данных по времени.
* `cnt` — общее количество новых аренд велосипедов (bike shares) за период между текущим и следующим `timestamp`. Обрати внимание, что это целевая переменная задачи регрессии.
* `t1` — фактическая температура воздуха (в градусах Цельсия).
* `t2` — температура «по ощущениям» (в градусах Цельсия); учитывает влияние ветра и влажности.
* `hum` — относительная влажность воздуха в процентах.
* `wind_speed` — скорость ветра (в километрах в час).
* `weather_code` — категориальный признак, описывающий погодные условия:
 * 1 — ясно;
 * 2 — рассеянная облачность / немного облаков;
 * 3 — переменная облачность (частично закрыто облаками);
 * 4 — пасмурно;
 * 7 — дождь / слабый ливень / мелкий дождь;
 * 10 — дождь с грозой;
 * 26 — снегопад;
 * 94 — туман / морозный туман.

* `is_holiday` — логический признак: 1 — праздничный день, 0 — обычный день.
* `is_weekend` — логический признак: 1 — выходной день (суббота или воскресенье), 0 — будний день.
* `season` — категориальный признак, обозначающий время года:
 * 0 — весна;
 * 1 — лето;
 * 2 — осень;
 * 3 — зима.



## Предобработка данных

Посмотрим на данные.

In [ ]:
bike.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17414 entries, 0 to 17413
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   timestamp     17414 non-null  object 
 1   cnt           17414 non-null  int64  
 2   t1            17414 non-null  float64
 3   t2            17414 non-null  float64
 4   hum           17414 non-null  float64
 5   wind_speed    17414 non-null  float64
 6   weather_code  17414 non-null  float64
 7   is_holiday    17414 non-null  float64
 8   is_weekend    17414 non-null  float64
 9   season        17414 non-null  float64
dtypes: float64(8), int64(1), object(1)
memory usage: 1.3+ MB


In [ ]:
bike.columns

Index(['timestamp', 'cnt', 't1', 't2', 'hum', 'wind_speed', 'weather_code',
       'is_holiday', 'is_weekend', 'season'],
      dtype='object')

In [ ]:
bike.describe().T

,count,mean,std,min,25%,50%,75%,max
cnt,17414.0,1143.101642,1085.108068,0.0,257.0,844.0,1671.75,7860.0
t1,17414.0,12.468091,5.571818,-1.5,8.0,12.5,16.00,34.0
t2,17414.0,11.520836,6.615145,-6.0,6.0,12.5,16.00,34.0
hum,17414.0,72.324954,14.313186,20.5,63.0,74.5,83.00,100.0
wind_speed,17414.0,15.913063,7.894570,0.0,10.0,15.0,20.50,56.5
weather_code,17414.0,2.722752,2.341163,1.0,1.0,2.0,3.00,26.0
is_holiday,17414.0,0.022051,0.146854,0.0,0.0,0.0,0.00,1.0
is_weekend,17414.0,0.285403,0.451619,0.0,0.0,0.0,1.00,1.0
season,17414.0,1.492075,1.118911,0.0,0.0,1.0,2.00,3.0


In [ ]:
# Есть ли дубли в строках

duplicate_rows = bike.duplicated().sum()
if duplicate_rows > 0:
    print("\nКоличество дублирующихся строк:", duplicate_rows)
    print("Дубли:")
    print(df[df.duplicated()])
else:
    print("\nДублирующиеся строки отсутствуют.")


Дублирующиеся строки отсутствуют.


In [ ]:
# Есть ли пропущенные значения

bike.isnull().sum()

,0
timestamp,0
cnt,0
t1,0
t2,0
hum,0
wind_speed,0
weather_code,0
is_holiday,0
is_weekend,0
season,0


Проведём небольшую предобработку для более удобной работы с данными.

In [ ]:
# Переименуем некоторые столбцы

bike.rename(columns={'cnt':'total_count', 't1':'temp', 't2':'feel_temp', 'hum':'humidity'}, inplace=True)
bike.columns

Index(['timestamp', 'total_count', 'temp', 'feel_temp', 'humidity',
       'wind_speed', 'weather_code', 'is_holiday', 'is_weekend', 'season'],
      dtype='object')

In [ ]:
# Разделим дату и время из столбца timestamp на несколько отдельных признаков, а именно:
# год, месяц, день, день недели (номер), час

bike.timestamp = pd.to_datetime(bike.timestamp)

bike['year'] = pd.to_datetime(bike['timestamp'], format='%d/%m/%Y').dt.year
bike['month'] = pd.to_datetime(bike['timestamp'], format='%d/%m/%Y').dt.month
bike['day'] = pd.to_datetime(bike['timestamp'], format='%d/%m/%Y').dt.day
bike['day_of_week'] = pd.to_datetime(bike['timestamp'], format='%d/%m/%Y').dt.dayofweek
bike['hour'] = pd.to_datetime(bike['timestamp'], format='%d/%m/%Y').dt.hour

In [ ]:
bike.head()

,timestamp,total_count,temp,feel_temp,humidity,wind_speed,weather_code,is_holiday,is_weekend,season,year,month,day,day_of_week,hour
0,2015-01-04 00:00:00,182,3.0,2.0,93.0,6.0,3.0,0.0,1.0,3.0,2015,1,4,6,0
1,2015-01-04 01:00:00,138,3.0,2.5,93.0,5.0,1.0,0.0,1.0,3.0,2015,1,4,6,1
2,2015-01-04 02:00:00,134,2.5,2.5,96.5,0.0,1.0,0.0,1.0,3.0,2015,1,4,6,2
3,2015-01-04 03:00:00,72,2.0,2.0,100.0,0.0,1.0,0.0,1.0,3.0,2015,1,4,6,3
4,2015-01-04 04:00:00,47,2.0,0.0,93.0,6.5,1.0,0.0,1.0,3.0,2015,1,4,6,4


Дальше ты будешь строить графики и делать по ним выводы. К первым задачам вывод уже написан — это поможет тебе понять, как их писать.

## Задача 1 [1 балл]

Для начала построй два графика.

1. Построй гистограмму распределения общего количества арендованных велосипедов (`total_count`) с наложенной линией KDE (Kernel Density Estimate), чтобы визуально оценить форму распределения данных. Оформи график с понятным заголовком и подписями осей. **[0,5 балла]**
  > **Примечание.** К этому графику вывод уже написан — ты найдёшь его ниже.
2. Преобразуй столбец `timestamp` к формату даты без времени (`date`). Объедини данные о прокате велосипедов по дням, посчитав суммарное количество арендованных велосипедов за каждый день. Построй линейный график с динамикой суточного числа аренд (`total_count`) во времени. Дополни график заголовком, подписью оси $Y$, а также поверни подписи по оси $X$ на 90 градусов для читаемости. **[0,5 балла]**

In [ ]:
# Напиши здесь код для гистограммы распределения

**Вывод**

Гистограмма показывает распределение количества аренд велосипедов. Видна смещённость вправо. У распределения выраженная правосторонняя асимметрия — большинство значений приходится на малые значения с заметным пиком в области очень низких показателей. Это говорит о том, что чаще всего велосипеды арендуют в небольшом количестве, а случаи массового проката встречаются значительно реже.

In [ ]:
# Напиши здесь код для линейного графика с динамикой total_count

## Задача 2 [3 балла]

Продолжи и построй более сложные графики.

###Задача 2.1 [1 балл]



1. Сгруппируй данные по столбцу `month` и посчитай суммарное количество аренд велосипедов по каждому месяцу.
2. Построй линейный график с зависимостью общего количества аренд от месяца.
3. Реализуй форматирование оси $Y$ так, чтобы значения отображались в тысячах.
4. Добавь подписи осей и заголовок графика.
5. Отобрази итоговый график.

Напиши вывод о сезонности аренды велосипедов по полученному графику.



In [ ]:
# Напиши здесь код для линейного графика с зависимостью общего количества аренд от месяца

**Твой вывод**

###Задача 2.2 [1 балл]

1. Сгруппируй данные по сезонам и рассчитай для каждого сезона максимальное, минимальное, среднее и суммарное количество аренд велосипедов с помощью функции `agg`.
2. Реализуй столбчатую диаграмму (barplot), которая показывает суммарное количество аренд велосипедов по сезонам. Используй значения суммы, полученные на предыдущем шаге.
3. Используй читаемые подписи сезонов вместо числовых значений (обрати внимание, как они закодированы в датасете).
4. Настрой размер графика (`figsize=(8, 4)`) и выбери цветовую палитру (`viridis`).
5. Добавь подписи осей `Season` и `Bike Shares` и заголовок графика `Total Bike Shares by Season`.
6. Отформатируй метки оси $Y$ так, чтобы значения были приведены в тысячах.
7. Выведи построенный график.

Напиши вывод: в какие сезоны суммарное количество аренд больше, а в какие — меньше.

In [ ]:
# Напиши здесь код для столбчатой диаграммы, показывающей общее количество аренд велосипедов по сезонам

**Твой вывод**

###Задача 2.3 [1 балл]

1. Сгруппируй датафрейм `bike` по признакам `season` и `day_of_week`. Вычисли среднее значение столбца `total_count` для каждой группы и сбрось индексы.
2. Построй `pointplot`, где по оси $X$ — `day_of_week`, по оси $Y$ — `total_count`. Цветом (`hue`) обозначь сезоны, заменив числовые коды на строковые метки (как и в задаче выше).
3. Задай подписи осей и заголовок графика.
4. Обнови подписи оси $X$ так, чтобы вместо чисел отображались названия дней недели.
5. Добавь легенду графика.
6. Используй `plt.tight_layout()` для корректного размещения элементов, а затем выведи график.

Дополнительно напиши вывод по полученному графику. Можешь использовать шаблон:

* пиковая нагрузка наблюдается ... особенно с ... по ...;
* на выходных отмечается ... по сравнению с рабочей неделей;
* на графике наблюдаются сезонные колебания, они ... нашу гипотезу, обозначенную выше;
* характер поведения данных ... во все сезоны.




In [ ]:
# Напиши код здесь для pointplot со средним количеством аренд по дням недели по сезонам

**Твой вывод**

## Задача 3 [2 балла]

###Задача 3.1 [1 балл]

1. Построй линейный график, который показывает распределение количества поездок на велосипедах в зависимости от времени суток.
2. Используй признак `is_weekend` в качестве группирующего фактора, чтобы сравнить популярность велопроката в будние и выходные дни.
3. Настрой легенду так, чтобы линии были подписаны как `No` (будний день) и `Yes` (выходной день).
4. Добавь заголовок графика.
5. Отобрази график с использованием функции `plt.tight_layout()` для корректного размещения элементов.
6. Напиши вывод. Опиши наблюдения по утренним и вечерним пикам, разницу между рабочими днями и выходными.



In [ ]:
# Напиши здесь код для линейного графика, показывающего распределение количества поездок на велосипедах в зависимости от часа суток по будням и выходным дням

**Твой вывод**

###Задача 3.2 [1 балл]
1. Создай график с двумя круговыми диаграммами, расположенными в одну строку.
2. Для первой диаграммы отобрази распределение количества поездок по признаку `is_holiday` с метками `No` и `Yes`. Укажи процентное соотношение каждого значения, используй цвета `lightblue` и `orange`. Установи начальный угол (`startangle`) на 90 градусов и добавь заголовок.
3. Для второй диаграммы отобрази распределение количества поездок по признаку `is_weekend` с метками `Weekday` и `Weekend`. Укажи процентное соотношение каждого значения, используй те же цвета и начальный угол, добавь заголовок.
4. Обеспечь компактное расположение графиков с помощью tight_layout.
5. Отобрази полученный график.
6. Опиши обе круговые диаграммы. Какие выводы  можно сделать по каждой из них?


In [ ]:
# Напиши здесь код для двух круговых диаграмм

**Твой вывод**

## Задача 4 [3 балла]

###Задача 4.1 **[1 балл]**

1. Построй два `scatter plot` на одной фигуре с двумя строками и одним столбцом, `figsize=(15, 10)`.
2. На первом графике отобрази зависимость количества поездок на велосипедах (`total_count`) от температуры (`temp`). Обязательно добавь заголовки.
3. На втором графике отобрази зависимость количества поездок на велосипедах (`total_count`) от кода погоды (`weather_code`). Также обязательно добавь заголовки.
 * Добавь на ось $X$ второго графика метки с названиями погодных условий согласно описанию датасета.
 * Поверни подписи на 90 градусов, выровняй по центру, задай размер шрифта 10.
4. Обеспечь удобное расположение графиков с помощью `plt.tight_layout(pad=3.0)`.
5. В выводе напиши, какие закономерности ты видишь на каждом из графиков.


In [ ]:
# Напиши здесь код для двух scatter plot

**Твой вывод**

###Задача 4.2 [2 балла]

In [ ]:
# Прежде чем перейти к следующему шагу — немного подправь данные
# Cтандартизация через StandardScaler

from sklearn.preprocessing import StandardScaler

# Возьми только числовые признаки
numerical_features = bike[['total_count', 'temp', 'feel_temp', 'humidity', 'wind_speed']]

# Инициализируй скейлер
scaler = StandardScaler()

# Примени его к числовым признакам
df_scaled = scaler.fit_transform(numerical_features)

# Переведи результат обратно в DataFrame
df_scaled = pd.DataFrame(df_scaled, columns=['total_count', 'temp', 'feel_temp', 'humidity', 'wind_speed'])



Создай финальный график.

1. Сделай расчёт корреляционной матрицы для датафрейма `df_scaled`. Построй тепловую карту (`heatmap`) этой матрицы с отображением значений корреляций, используй цветовую палитру `coolwarm`. Добавь заголовок и отобрази график.

2. Напиши подробный вывод по получившейся матрице. В первую очередь нас интересует, с чем коррелирует целевая переменная `total_count` и наибольшие значения положительной и отрицательной корреляции в матрице. Напиши минимум три пункта, которые вызывают интерес и могут быть полезны для решения задачи.



In [ ]:
# Напиши код здесь — матрица корреляции

**Твой вывод**

## Задача 5 [3 балла]

В задаче 5 ты полностью подготовишь данные для подачи в модель: выполнишь нормализацию признаков и разделишь данные на обучающую и тестовую выборки.  

1. Обучи любую модель регрессии из библиотеки `scikit-learn`.
2. Реализуй собственные функции для расчёта коэффициента детерминации (R²) и среднеквадратичной ошибки (MSE). **[1 балл]**
3. Вычисли значения метрик с помощью своих функций и сравни их с результатами расчёта с использованием функций из `scikit-learn` (должны получиться одинаковыми).
4. Составь краткий вывод по метрикам: какие значения получились и как их можно интерпретировать с точки зрения качества модели. **[2 балла]**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
X = bike.drop(['total_count','timestamp'], axis=1)
y = bike['total_count']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.concat(
    [
        pd.DataFrame(
            scaler.fit_transform(X_train[['temp', 'feel_temp', 'humidity', 'wind_speed']]),
            columns=['temp', 'feel_temp', 'humidity', 'wind_speed']
        ).reset_index(drop=True),
        X_train[['weather_code', 'is_holiday', 'is_weekend', 'season', 'year', 'month','day','day_of_week','hour']].reset_index(drop=True)
    ],
    axis=1
)
X_test_scaled = pd.concat(
    [
        pd.DataFrame(
            scaler.transform(X_test[['temp', 'feel_temp', 'humidity', 'wind_speed']]),
            columns=['temp', 'feel_temp', 'humidity', 'wind_speed']
        ).reset_index(drop=True),
        X_test[['weather_code', 'is_holiday', 'is_weekend', 'season', 'year', 'month','day','day_of_week','hour']].reset_index(drop=True)
    ],
    axis=1
)

In [ ]:
# Напиши здесь код для модели, метрик и валидации

**Твой вывод**